In [ ]:
# Databricks notebook source
# 03_clean_gd

import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType

CATALOG = "decide_catalog"
SCHEMA = "decide_schema"
VOLUME_PATH = "/Volumes/decide_catalog/decide_schema/decide_volume"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")


def should_run_pipeline():
    try:
        value = dbutils.jobs.taskValues.get(taskKey="00_check_source_changes", key="should_run", default="true")
        return str(value).lower() == "true"
    except Exception:
        return True


def source_path(file_name):
    return f"{VOLUME_PATH}/{file_name}"


def sha256_hash(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def month_start(values):
    return pd.to_datetime(values, errors="coerce").dt.to_period("M").dt.start_time.dt.date


def week_start(values, week_start="sunday"):
    dates = pd.to_datetime(values, errors="coerce")
    freq = "W-SAT" if week_start == "sunday" else "W-SUN"
    return dates.dt.to_period(freq).dt.start_time.dt.date


def max_with_na(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().any():
        return values.max(skipna=True)
    return np.nan


def write_delta(pdf, table_name):
    output_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Date",
        "Date_month",
        "Date_week",
        "Pathogen",
        "Result",
    ]
    pdf = pdf.reindex(columns=output_cols).copy()

    string_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Pathogen",
    ]
    for col in string_cols:
        pdf[col] = pdf[col].where(pd.notna(pdf[col]), None).astype(object)
    for col in ["Date", "Date_month", "Date_week"]:
        pdf[col] = pd.to_datetime(pdf[col], errors="coerce").dt.date
    pdf["Result"] = pd.to_numeric(pdf["Result"], errors="coerce")

    schema = StructType(
        [
            StructField("Lab_reference", StringType(), True),
            StructField("Country", StringType(), True),
            StructField("Breed", StringType(), True),
            StructField("Province", StringType(), True),
            StructField("Farm_ID", StringType(), True),
            StructField("Diagnostic_test", StringType(), True),
            StructField("Sample_type", StringType(), True),
            StructField("Samplenumber", StringType(), True),
            StructField("Date", DateType(), True),
            StructField("Date_month", DateType(), True),
            StructField("Date_week", DateType(), True),
            StructField("Pathogen", StringType(), True),
            StructField("Result", DoubleType(), True),
        ]
    )
    sdf = spark.createDataFrame(pdf, schema=schema)
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Wrote {sdf.count()} rows to {full_name}")

if not should_run_pipeline():
    dbutils.notebook.exit("No source changes detected; skipping GD")

raw = pd.read_excel(source_path("250808_data_RGD_DECIDE.xlsx"), engine="openpyxl")
df = raw.rename(columns={"Dossier_ID": "Filenumber", "sample_id": "Samplenumber", "farm_ID": "Farm_ID", "project": "Project", "date": "Date"})
df["Country"] = "The Netherlands"
df["Lab_reference"] = "2"
df["Sample_type"] = np.select([df["reason_of_sampling"].eq("Autopsy"), df["sample"].eq("BAL"), df["sample"].eq("SWABS"), df["sample"].eq("OTHER")], ["Autopsy", "BAL", "Swab", "Unknown"], default="Missing")
df["Diagnostic_test"] = df["test"].map({"PCR": "PCR", "Kweek": "Culture"}).fillna("Missing")
df["Breed"] = df["breed"].map({"beef": "Beef", "dairy": "Dairy", "mixed": "Mixed", "veal": "Veal", "other": "Unknown", "rearing": "Unknown", "unknown": "Unknown"}).fillna("Unknown")
df["Province"] = df["provincie"].map({"DR": "Drenthe", "FL": "Flevoland", "FR": "Friesland", "GL": "Gelderland", "GR": "Groningen", "LB": "Limburg", "NB": "North Brabant", "NH": "North Holland", "OV": "Overijssel", "UT": "Utrecht", "ZH": "South Holland", "ZL": "Zeeland"}).fillna("Missing")
pathogen_cols = ["PM", "MH", "HS", "MB", "BRSV", "PI3", "BCV"]
df = df[["Filenumber", "Diagnostic_test", "Samplenumber", "Country", "Lab_reference", "Sample_type", "Breed", *pathogen_cols, "Date", "Province", "Project", "Farm_ID"]].drop_duplicates()
for col in ["Filenumber", "Samplenumber", "Farm_ID"]:
    df[col] = df[col].apply(sha256_hash)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
df["Date_month"] = month_start(df["Date"])
df["Date_week"] = week_start(df["Date"], week_start="sunday")
df = df[df["Project"].isin(["monitoring", "no project"])]
group_cols = ["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Samplenumber", "Date_month", "Date_week", "Date"]
grouped = df.groupby(group_cols, dropna=False)[pathogen_cols].agg(max_with_na).reset_index()
barometer = grouped.melt(id_vars=group_cols, value_vars=pathogen_cols, var_name="Pathogen", value_name="Result")
barometer = barometer[["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Samplenumber", "Date", "Date_month", "Date_week", "Pathogen", "Result"]]
write_delta(barometer, "barometer_gd")
